# Donor growth — CRISP-DM walkthrough

Run cells **top to bottom** as we add each phase.

**Six phases (one sentence each):**

1. **Business understanding** — What decision does predicting “growth” support?  
2. **Data understanding** — What CSVs in `Dataset/` exist, how they join, and what they imply for the target?  
3. **Data preparation** — Which columns are inputs vs the target (no circular features)?  
4. **Modeling** — Which regressor and validation strategy (small *N* = 60)?  
5. **Evaluation** — Error metrics, residuals, overfitting check.  
6. **Deployment** — Save `growth_pipeline_v1.sav` and call `POST /growth/predict` in FastAPI.

We build **one phase at a time**; confirm each phase before the next.

## Phase 1: Business understanding

**Decision the model supports:** Prioritize supporters by **estimated lifetime giving scale** (dollar footprint) so fundraising can focus stewardship and asks where potential impact is higher.

**Target (regression):** `total_monetary_value` — supporter-level sum of estimated giving (we can **define** it from `donations.csv` / allocations, or **read** it from the engineered rollup in `Created .csv for Pipelines/donor_and_potential_growth.csv`). Either way, the **target is not** an input feature.

**Inputs (principle):** Signals that explain giving **without** leaking the target: e.g. frequency, recency, avg gift, channel mix, recurring behavior, program mix, in-kind patterns, optional aggregates from **`supporters.csv`**, **`donations.csv`**, **`donation_allocations.csv`**, **`in_kind_donation_items.csv`**, and **`social_media_posts.csv`** (via `referral_post_id` on gifts). Final feature list and joins are decided in **Phase 3**.

**Data sources:** All under `Dataset/lighthouse_csv_v7/` (plus the engineered subfolder). We are **not** limited to a single CSV—Phase 2 maps what is available and how keys connect.

**Modeling grain:** One row per `supporter_id` (~60 supporters); raw tables are transactional and are aggregated or joined in preparation.

**Success in business terms:** Useful **ranking** and **rough** dollar bands for planning; with small *N*, stress **cross-validated** error and uncertainty.

---

**Checkpoint — Phase 1:** Approve this framing before we lock Phase 3 feature engineering details.

## Phase 2: Data understanding

**What we do here:** Inventory **helpful CSVs** under `Dataset/lighthouse_csv_v7/`, their **row counts**, **keys**, and how they **relate** to predicting supporter-level `total_monetary_value`. The next cell loads the main tables and summarizes join paths (no modeling yet).

**Talking points for video:** Gifts roll up to supporters via `supporter_id`; allocations and in-kind items hang off `donation_id`; gifts can link to social posts via `referral_post_id`. Other files (education, residents, safehouses, etc.) are **context** for the nonprofit story—we call out which are **directly gift-linked** vs **peripheral** for this growth use case.

**Checkpoint — Phase 2:** After you run the code and skim the output, approve before **Phase 3: Data preparation** (define *y*, build *X*, cleaning).

In [1]:
# -----------------------------------------------------------------------------
# Phase 2 — Data understanding: which CSVs exist and how they connect
# -----------------------------------------------------------------------------
# Goal: see row counts, join keys (supporter_id, donation_id), and how raw gifts
# relate to the engineered growth/retention table before we build X and y later.
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
_repo = None
for _p in [_here, *_here.parents]:
    if (_p / "Dataset" / "lighthouse_csv_v7").is_dir():
        _repo = _p
        break
if _repo is None:
    raise FileNotFoundError(
        "Could not find Dataset/lighthouse_csv_v7/. "
        "Place data under <repo>/Dataset/lighthouse_csv_v7/ and open the `keeper` folder in Cursor. "
        f"Searched from cwd={_here}"
    )
DATA = _repo / "Dataset" / "lighthouse_csv_v7"
ENG = DATA / "Created .csv for Pipelines" / "donor_and_potential_growth.csv"

# Files most directly useful for modeling "giving scale" (Phase 3+ can add joins)
paths = {
    "engineered_donor_growth": ENG,
    "supporters": DATA / "supporters.csv",
    "donations": DATA / "donations.csv",
    "donation_allocations": DATA / "donation_allocations.csv",
    "in_kind_donation_items": DATA / "in_kind_donation_items.csv",
    "social_media_posts": DATA / "social_media_posts.csv",
}

print("=== File presence & row counts ===")
for label, path in paths.items():
    if path.is_file():
        # Only read first column to count rows quickly
        n = len(pd.read_csv(path, usecols=[0]))
        print(f"  {label}: {path.name} — {n} rows")
    else:
        print(f"  {label}: MISSING {path}")

# Full loads for relationship checks (these files are small/medium)
sup = pd.read_csv(paths["supporters"])
don = pd.read_csv(paths["donations"])
eng_df = pd.read_csv(paths["engineered_donor_growth"])
alloc = pd.read_csv(paths["donation_allocations"])
inkind = pd.read_csv(paths["in_kind_donation_items"])

print("\n=== Keys & overlap ===")
print(f"supporters: {len(sup)} rows, supporter_id range {sup['supporter_id'].min()}–{sup['supporter_id'].max()}")
print(f"donations: {len(don)} rows, unique supporters {don['supporter_id'].nunique()}")
print(f"engineered: {len(eng_df)} rows, target total_monetary_value describe:\n{eng_df['total_monetary_value'].describe()}")

print(f"\ndonation_allocations: {len(alloc)} rows, donation_id overlap with donations {alloc['donation_id'].isin(don['donation_id']).mean():.1%}")
print(f"in_kind_donation_items: {len(inkind)} rows, donation_id overlap {inkind['donation_id'].isin(don['donation_id']).mean():.1%}")

# Social attribution: gifts may reference a post that drove the gift
if "referral_post_id" in don.columns:
    ref = don["referral_post_id"]
    n_ref = int(ref.notna().sum())
    print(f"\ndonations with referral_post_id: {n_ref} / {len(don)}")
    if n_ref and paths["social_media_posts"].is_file():
        smp = pd.read_csv(paths["social_media_posts"], usecols=["post_id"])
        post_ids = set(smp["post_id"].values)
        linked = don.loc[ref.notna(), "referral_post_id"].astype(int).isin(post_ids).sum()
        print(f"  referral_post_id values found in social_media_posts.post_id: {linked} / {n_ref}")

print("\n=== Peripheral tables (optional for future features / storytelling) ===")
peripheral = sorted(
    x.name
    for x in DATA.glob("*.csv")
    if x.name
    not in {
        "supporters.csv",
        "donations.csv",
        "donation_allocations.csv",
        "in_kind_donation_items.csv",
        "social_media_posts.csv",
    }
)
print(", ".join(peripheral[:10]), "…" if len(peripheral) > 10 else "")


=== File presence & row counts ===
  engineered_donor_growth: donor_and_potential_growth.csv — 60 rows
  supporters: supporters.csv — 60 rows
  donations: donations.csv — 420 rows
  donation_allocations: donation_allocations.csv — 521 rows
  in_kind_donation_items: in_kind_donation_items.csv — 129 rows
  social_media_posts: social_media_posts.csv — 812 rows

=== Keys & overlap ===
supporters: 60 rows, supporter_id range 1–60
donations: 420 rows, unique supporters 59
engineered: 60 rows, target total_monetary_value describe:
count       60.000000
mean      4895.130167
std       3571.012801
min          0.000000
25%       2161.717500
50%       3926.685000
75%       6871.930000
max      14240.290000
Name: total_monetary_value, dtype: float64

donation_allocations: 521 rows, donation_id overlap with donations 100.0%
in_kind_donation_items: 129 rows, donation_id overlap 100.0%

donations with referral_post_id: 77 / 420
  referral_post_id values found in social_media_posts.post_id: 77 / 77



## Phase 3: Data preparation

**What we do here:** Fix **y** = `total_monetary_value` (from the engineered rollup—aligned with Phase 2). Build **X** only from columns that are **not** the target: RFM-style numerics plus `top_program_interest`. **`total_monetary_value` is never a column in X** (no circular regression).

**Cleaning:** Same rules as the retention notebook for overlapping fields (missing recency → cohort max; zero-gift avg → 0; missing program → `Unknown`).

**Preprocessing (sklearn):** Median imputer + scaler on numerics; most-frequent + one-hot on the program column. The transformer is fit **only on training folds** inside CV in Phase 4 (here we only show an exploratory `fit_transform` on all rows for shape check).

**Scope note:** Phase 2’s extra CSVs (allocations, in-kind, raw gifts) are available for **future** feature engineering; this phase locks a **baseline** feature set so we can ship a pipeline and FastAPI endpoint on schedule. We can extend `X` later without changing the CRISP-DM story.

**Checkpoint — Phase 3:** Approve this *X* / *y* definition before **Phase 4: Modeling**.

In [2]:
# -----------------------------------------------------------------------------
# Phase 3 — Data preparation: define y, build X, clean, sklearn preprocessor
# -----------------------------------------------------------------------------
# Target y = total_monetary_value (we predict lifetime estimated giving scale).
# Features X must NOT include total_monetary_value (would be cheating).
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

_here = Path.cwd().resolve()
_repo = None
for _p in [_here, *_here.parents]:
    if (_p / "Dataset" / "lighthouse_csv_v7").is_dir():
        _repo = _p
        break
if _repo is None:
    raise FileNotFoundError(
        "Could not find Dataset/lighthouse_csv_v7/. "
        "Place data under <repo>/Dataset/lighthouse_csv_v7/ and open the `keeper` folder in Cursor. "
        f"Searched from cwd={_here}"
    )
ENG_PATH = _repo / "Dataset" / "lighthouse_csv_v7" / "Created .csv for Pipelines" / "donor_and_potential_growth.csv"

# Columns fed to the model (baseline set; Phase 2 tables available for future extras)
GROWTH_NUMERIC_FEATURES = [
    "recency_days",
    "frequency",
    "avg_monetary_value",
    "social_referral_count",
    "is_recurring_donor",
]
GROWTH_CATEGORICAL_FEATURES = ["top_program_interest"]
TARGET_GROWTH = "total_monetary_value"
GROWTH_FEATURE_COLUMNS = GROWTH_NUMERIC_FEATURES + GROWTH_CATEGORICAL_FEATURES


def clean_engineered_donor_df(df: pd.DataFrame) -> pd.DataFrame:
    """Align missing-value handling with retention notebook / FastAPI services."""
    out = df.copy()
    # No last-gift date → treat as very stale using cohort max observed recency
    max_recency = out["recency_days"].max(skipna=True)
    if pd.isna(max_recency):
        max_recency = 0.0
    out.loc[out["recency_days"].isna(), "recency_days"] = float(max_recency)

    zero_gift = out["frequency"].fillna(0) == 0
    out.loc[out["avg_monetary_value"].isna() & zero_gift, "avg_monetary_value"] = 0.0
    med_avg = out["avg_monetary_value"].median(skipna=True)
    if pd.isna(med_avg):
        med_avg = 0.0
    out["avg_monetary_value"] = out["avg_monetary_value"].fillna(med_avg)

    out["social_referral_count"] = out["social_referral_count"].fillna(0.0)

    out["top_program_interest"] = (
        out["top_program_interest"].fillna("Unknown").astype(str).str.strip()
    )
    out.loc[out["top_program_interest"] == "", "top_program_interest"] = "Unknown"

    return out


def assert_growth_frame_ready(df: pd.DataFrame) -> None:
    """Fail fast if required columns missing or still null after cleaning."""
    need = GROWTH_FEATURE_COLUMNS + [TARGET_GROWTH]
    missing = [c for c in need if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"
    assert TARGET_GROWTH not in GROWTH_FEATURE_COLUMNS
    nulls = df[GROWTH_FEATURE_COLUMNS + [TARGET_GROWTH]].isnull().sum()
    bad = nulls[nulls > 0]
    assert bad.empty, f"Nulls after clean:\n{bad}"


def build_growth_preprocessor() -> ColumnTransformer:
    """Numeric: median impute + scale. Categorical: mode impute + one-hot."""
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, GROWTH_NUMERIC_FEATURES),
            ("cat", categorical_pipe, GROWTH_CATEGORICAL_FEATURES),
        ]
    )


raw = pd.read_csv(ENG_PATH)
df_clean = clean_engineered_donor_df(raw)
assert_growth_frame_ready(df_clean)

X_growth = df_clean[GROWTH_FEATURE_COLUMNS]
y_growth = df_clean[TARGET_GROWTH]

print("Rows:", len(df_clean), "| X columns:", GROWTH_FEATURE_COLUMNS)
print("Target:", TARGET_GROWTH, "| y stats:\n", y_growth.describe())
print("\nConfirm target NOT in X:", TARGET_GROWTH not in X_growth.columns)

# Exploratory: fit preprocessor on ALL rows only to see matrix width (Phase 4 uses CV)
growth_preprocessor = build_growth_preprocessor()
X_prep_explore = growth_preprocessor.fit_transform(X_growth)
print("\nExploratory design matrix shape (fit on all rows):", X_prep_explore.shape)

display(df_clean[["supporter_id"] + GROWTH_FEATURE_COLUMNS + [TARGET_GROWTH]].head())


Rows: 60 | X columns: ['recency_days', 'frequency', 'avg_monetary_value', 'social_referral_count', 'is_recurring_donor', 'top_program_interest']
Target: total_monetary_value | y stats:
 count       60.000000
mean      4895.130167
std       3571.012801
min          0.000000
25%       2161.717500
50%       3926.685000
75%       6871.930000
max      14240.290000
Name: total_monetary_value, dtype: float64

Confirm target NOT in X: True

Exploratory design matrix shape (fit on all rows): (60, 12)


,supporter_id,recency_days,frequency,avg_monetary_value,social_referral_count,is_recurring_donor,top_program_interest,total_monetary_value
0,1,10.0,12.0,750.002500,3.0,1,Wellbeing,9000.03
1,2,297.0,4.0,969.340000,1.0,0,Education,3877.36
2,3,169.0,16.0,778.008125,1.0,1,Maintenance,12448.13
3,4,0.0,11.0,903.147273,3.0,1,Wellbeing,9934.62
4,5,150.0,5.0,950.234000,1.0,0,Transport,4751.17


## Phase 4: Modeling

**What we do here:** Wrap **`build_growth_preprocessor()`** and a **regressor** in one `Pipeline`. Compare several algorithms with the **same 5-fold CV** (`KFold`, shuffled—regression has no classes to stratify). Pick the model with the **lowest mean CV MAE** (mean absolute error in dollars); tie-break on higher mean CV R².

**Talking points:** Each fold refits scaling and one-hot on **training rows only**. With *N* = 60, keep trees/boosting **shallow / regularized**. After selection, we **fit the winner on all rows** for saving in Phase 6.

**Checkpoint — Phase 4:** Approve before **Phase 5: Evaluation** (residuals, train vs CV gap).

In [3]:
# -----------------------------------------------------------------------------
# Phase 4 — Modeling: compare regressors; pick lowest CV MAE (tie-break R²)
# -----------------------------------------------------------------------------
# Each Pipeline refits preprocessing inside each CV fold (no leakage).
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

# Requires Phase 3: X_growth, y_growth, build_growth_preprocessor()
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"neg_mae": "neg_mean_absolute_error", "r2": "r2"}

# Shallow / regularized models — only 60 rows
candidate_regressors = {
    "gradient_boosting": GradientBoostingRegressor(
        random_state=42,
        max_depth=3,
        n_estimators=120,
        learning_rate=0.08,
        subsample=0.9,
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        random_state=42,
        max_depth=4,
        max_iter=200,
        learning_rate=0.06,
        min_samples_leaf=3,
    ),
    "random_forest": RandomForestRegressor(
        random_state=42,
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=4,
    ),
    "ridge": Ridge(alpha=5.0),
}

rows = []
cv_by_name = {}
for name, reg in candidate_regressors.items():
    pipe = Pipeline(
        steps=[("prep", build_growth_preprocessor()), ("reg", clone(reg))]
    )
    scores = cross_validate(
        pipe,
        X_growth,
        y_growth,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
    )
    cv_by_name[name] = scores
    rows.append(
        {
            "model": name,
            "cv_mae_mean": -scores["test_neg_mae"].mean(),
            "cv_mae_std": scores["test_neg_mae"].std(),
            "cv_r2_mean": scores["test_r2"].mean(),
            "cv_r2_std": scores["test_r2"].std(),
        }
    )

leaderboard = pd.DataFrame(rows).sort_values(
    ["cv_mae_mean", "cv_r2_mean"], ascending=[True, False]
)
print("=== CV leaderboard (lower MAE is better; tie-break higher R²) ===")
display(leaderboard)

best_name = leaderboard.iloc[0]["model"]
growth_cv_scores = cv_by_name[best_name]
growth_model_name = best_name
growth_pipeline = Pipeline(
    steps=[
        ("prep", build_growth_preprocessor()),
        ("reg", clone(candidate_regressors[best_name])),
    ]
)

print(f"\nSelected model: {best_name}\n")
print("CV MAE (mean ± std):", f"{-growth_cv_scores['test_neg_mae'].mean():.1f} ± {growth_cv_scores['test_neg_mae'].std():.1f}")
print("CV R²  (mean ± std):", f"{growth_cv_scores['test_r2'].mean():.3f} ± {growth_cv_scores['test_r2'].std():.3f}")
print(
    "Train MAE / R² (same folds, for overfitting hint):",
    f"{-growth_cv_scores['train_neg_mae'].mean():.1f}",
    "/",
    f"{growth_cv_scores['train_r2'].mean():.3f}",
)

# Final fit on all rows — artifact for Phase 6 / FastAPI
growth_pipeline.fit(X_growth, y_growth)
print(f"\nFitted `growth_pipeline` ({best_name}) on all {len(X_growth)} rows (Phase 6 save).")


=== CV leaderboard (lower MAE is better; tie-break higher R²) ===


,model,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std
1,hist_gradient_boosting,803.064281,214.558390,0.858535,0.096217
0,gradient_boosting,812.958712,414.263511,0.857449,0.105998
3,ridge,1044.545050,258.663658,0.808659,0.049987
2,random_forest,1050.058306,359.834444,0.776630,0.117326



Selected model: hist_gradient_boosting

CV MAE (mean ± std): 803.1 ± 214.6
CV R²  (mean ± std): 0.859 ± 0.096
Train MAE / R² (same folds, for overfitting hint): 50.8 / 0.999

Fitted `growth_pipeline` (hist_gradient_boosting) on all 60 rows (Phase 6 save).


## Phase 5: Evaluation

**What we do here:** Measure how well the **chosen regressor** estimates `total_monetary_value` using **out-of-fold (OOF)** predictions — the same idea as validation, but every row gets a prediction from a model that did **not** train on that row in that fold.

**Metrics in plain English**

| Metric | Meaning |
|--------|--------|
| **MAE** | Typical prediction error in **dollars** (average \|actual − predicted\|). |
| **RMSE** | Like MAE, but **large errors count extra** (root mean squared error). |
| **R²** | How much of the ups-and-downs in giving the model explains (on this sample). |
| **OOF vs train** | If **train** error is much smaller than **OOF**, the model is **memorizing** this table somewhat. |

**Plots:** (1) actual vs OOF predicted — points near the red line are accurate. (2) residuals — ideally random cloud around zero.

**Tables:** largest **OOF** mistakes and highest **actual** donors (sanity check).

**Checkpoint:** Phase 6 saves the model and connects to **FastAPI** `POST /growth/predict`.


In [4]:
# -----------------------------------------------------------------------------
# Phase 5 — Evaluation: plain-language metrics + plots + feature signal
# -----------------------------------------------------------------------------
# OOF = "out of fold": each row predicted only when it was in the validation fold.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# Re-run CV predictions with a fresh clone each fold (sklearn handles this)
y_oof = cross_val_predict(growth_pipeline, X_growth, y_growth, cv=cv)

mae_oof = mean_absolute_error(y_growth, y_oof)
rmse_oof = root_mean_squared_error(y_growth, y_oof)
r2_oof = r2_score(y_growth, y_oof)

# Predictions after fitting on ALL rows (can be tighter than OOF = overfitting signal)
y_hat_train = growth_pipeline.predict(X_growth)
mae_train = mean_absolute_error(y_growth, y_hat_train)
r2_train = r2_score(y_growth, y_hat_train)

cv_mae_mean = float(-growth_cv_scores["test_neg_mae"].mean())
cv_mae_std = float(growth_cv_scores["test_neg_mae"].std())
y_mean = float(y_growth.mean())

print("=== How to read these numbers ===")
print(
    "• MAE  = average |error| in dollars when we pretend each row was 'new' (OOF).\n"
    "• RMSE = punishes big misses more than MAE.\n"
    "• R²   = share of variance explained (1.0 = perfect on this sample; can be negative on bad folds).\n"
    "• OOF  = honest for this small dataset; 'train' line uses the model that already saw every row.\n"
)

print("=== Out-of-fold (honest) performance ===")
print(f"MAE:  ${mae_oof:,.2f}  (~{100 * mae_oof / y_mean:.1f}% of mean giving ${y_mean:,.0f})")
print(f"RMSE: ${rmse_oof:,.2f}")
print(f"R²:   {r2_oof:.3f}")

print("\n=== Same as Phase 4 aggregated CV scores ===")
print(f"CV MAE (mean ± std): ${cv_mae_mean:,.2f} ± ${cv_mae_std:,.2f}")

print("\n=== Overfitting check ===")
print(f"Train MAE (saw all rows): ${mae_train:,.2f}")
print(f"OOF MAE (did not see row): ${mae_oof:,.2f}")
gap_mae = mae_oof - mae_train
print(f"Gap (OOF − train MAE):    ${gap_mae:,.2f}  → positive means model fits training tighter than it generalizes.")
print(f"Train R²: {r2_train:.3f}  |  OOF R²: {r2_oof:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
ax0, ax1 = axes
ax0.scatter(y_growth, y_oof, alpha=0.75, edgecolors="k", linewidths=0.3)
lims = [min(y_growth.min(), y_oof.min()), max(y_growth.max(), y_oof.max())]
ax0.plot(lims, lims, "r--", lw=1, label="perfect")
ax0.set_xlabel("Actual total_monetary_value")
ax0.set_ylabel("OOF predicted")
ax0.set_title(f"Actual vs OOF ({growth_model_name})")
ax0.legend()
ax0.set_aspect("equal", adjustable="box")

resid = y_growth.values - y_oof
ax1.axhline(0, color="r", linestyle="--", lw=1)
ax1.scatter(y_oof, resid, alpha=0.75, edgecolors="k", linewidths=0.3)
ax1.set_xlabel("OOF predicted")
ax1.set_ylabel("Residual (actual − pred)")
ax1.set_title("Residuals (OOF)")
plt.tight_layout()
plt.show()

print("\n=== Who drives error? (largest |OOF residual|) ===")
cmp = pd.DataFrame(
    {
        "supporter_id": df_clean["supporter_id"].values,
        "actual": y_growth.values,
        "pred_oof": y_oof,
        "pred_train_fit": y_hat_train,
    }
)
cmp["oof_error"] = cmp["actual"] - cmp["pred_oof"]
cmp["abs_oof_error"] = cmp["oof_error"].abs()
display(cmp.sort_values("abs_oof_error", ascending=False).head(8))
print("=== Highest actual giving (sanity check) ===")
display(cmp.sort_values("actual", ascending=False).head(8))

prep = growth_pipeline.named_steps["prep"]
reg = growth_pipeline.named_steps["reg"]
feat_names = prep.get_feature_names_out()

print(f"\n=== Which inputs mattered? ({growth_model_name}) ===")
if hasattr(reg, "feature_importances_"):
    imp = pd.Series(reg.feature_importances_, index=feat_names).sort_values(ascending=False)
    display(imp.to_frame("importance"))
elif hasattr(reg, "coef_"):
    imp = pd.Series(np.abs(reg.coef_).ravel(), index=feat_names).sort_values(ascending=False)
    display(imp.to_frame("abs_coef"))
else:
    print("No feature_importances_ or coef_ for this estimator.")


ModuleNotFoundError: No module named 'matplotlib'

## Phase 6: Deployment

**What we do here:** Save the fitted **`growth_pipeline`** to **`pipelines/growth_pipeline_v1.sav`** (same folder as retention). The FastAPI app loads it automatically when the file exists (`GET /health` shows `growth_pipeline_loaded`). Endpoint: **`POST /growth/predict`** (see `/docs`).

**The printed summary** restates what the file contains, OOF vs train error, and example ranked predictions — in wording you can reuse in a report or video.


In [ ]:
# -----------------------------------------------------------------------------
# Phase 6 — Save artifact + human-readable summary for reports / API handoff
# -----------------------------------------------------------------------------
import joblib
import numpy as np
from pathlib import Path

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_predict

_here = Path.cwd().resolve()
_ml_pipelines_root = None
for _p in [_here, *_here.parents]:
    if (_p / "app" / "main.py").is_file() and (_p / "requirements.txt").is_file():
        _ml_pipelines_root = _p
        break
    _mp = _p / "ml-pipelines"
    if (_mp / "app" / "main.py").is_file():
        _ml_pipelines_root = _mp
        break
if _ml_pipelines_root is None:
    raise FileNotFoundError(
        "Could not find ml-pipelines/ (folder with app/main.py). "
        f"Searched from cwd={_here}"
    )
ARTIFACT_PATH = _ml_pipelines_root / "pipelines" / "growth_pipeline_v1.sav"
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(growth_pipeline, ARTIFACT_PATH)
loaded = joblib.load(ARTIFACT_PATH)
_same = np.allclose(loaded.predict(X_growth), growth_pipeline.predict(X_growth), rtol=1e-5)
print(f"Saved growth pipeline to:\n  {ARTIFACT_PATH}")
print(f"Reload sanity check (predictions match): {_same}\n")

# Recompute OOF for narrative (same cv as Phases 4–5)
y_oof_6 = cross_val_predict(growth_pipeline, X_growth, y_growth, cv=cv)
mae_oof = mean_absolute_error(y_growth, y_oof_6)
r2_oof = r2_score(y_growth, y_oof_6)
y_hat = growth_pipeline.predict(X_growth)
mae_tr = mean_absolute_error(y_growth, y_hat)
r2_tr = r2_score(y_growth, y_hat)
cv_mae = float(-growth_cv_scores["test_neg_mae"].mean())

print("=" * 60)
print("WHAT THIS FILE IS")
print("=" * 60)
print(
    f"• sklearn Pipeline: preprocessing + **{growth_model_name}** regressor.\n"
    "• Predicts **total_monetary_value** from the Phase 3 feature columns only.\n"
    "• Trained on all 60 supporters after model comparison in Phase 4.\n"
    "• FastAPI: POST /growth/predict with JSON keys matching GROWTH_FEATURE_COLUMNS.\n"
)

print("=" * 60)
print("HOW GOOD IS IT? (same metrics as Phase 5)")
print("=" * 60)
print(f"• OOF MAE ≈ ${mae_oof:,.0f}  |  CV mean test MAE ≈ ${cv_mae:,.0f}")
print(f"• OOF R²  = {r2_oof:.3f}")
print(f"• Train MAE ${mae_tr:,.0f} vs OOF MAE ${mae_oof:,.0f} → gap shows optimism on training rows.")

if mae_tr < mae_oof * 0.7:
    over = "Train error much lower than OOF — treat dollar predictions as rough rankings, not exact budgets."
elif mae_oof - mae_tr > 200:
    over = "Moderate gap — some overfitting likely with N=60."
else:
    over = "Train and OOF errors in the same ballpark (still only 60 rows)."
print(f"• Overfitting read: {over}\n")

print("=" * 60)
print("EXAMPLE — estimated giving rank (train-fit; illustration only)")
print("=" * 60)
ranked = (
    df_clean[["supporter_id"]]
    .assign(actual_total=y_growth.values, predicted_total=y_hat)
    .assign(error=lambda d: d["actual_total"] - d["predicted_total"])
    .sort_values("predicted_total", ascending=False)
)
display(ranked.head(8))


## Pipeline summary

This notebook builds an **end-to-end donor “growth” regressor** from the engineered supporter table `donor_and_potential_growth.csv` (one row per supporter under `Dataset/lighthouse_csv_v7/Created .csv for Pipelines/`).

**Goal:** Estimate **`total_monetary_value`** (supporter-level giving scale) so fundraising can **prioritize stewardship and asks** by rough dollar bands and ranking. The target is **never** included in **X** (no circular regression).

**Flow:**

1. **Load & explore (Phase 2)** — Inventory CSVs under `Dataset/lighthouse_csv_v7/`, row counts, keys (`supporter_id`, `donation_id`, referrals to `social_media_posts`), and how raw gifts relate to the engineered rollup. Extra tables (allocations, in-kind, etc.) are documented for **future** features; the shipped baseline uses the engineered file only.
2. **Prepare features (Phase 3)** — **y** = `total_monetary_value`. **X** = six inputs: `recency_days`, `frequency`, `avg_monetary_value`, `social_referral_count`, `is_recurring_donor`, and `top_program_interest`. Rows are cleaned (aligned with the retention notebook / FastAPI), then a **`ColumnTransformer`**: median imputation + **`StandardScaler`** on numerics; most-frequent + **`OneHotEncoder`** on program.
3. **Model selection (Phase 4)** — Four regressors (gradient boosting, histogram gradient boosting, random forest, ridge) are compared with **5-fold `KFold` CV** inside a single **`Pipeline`** so preprocessing refits only on training folds. The winner minimizes **mean CV MAE** (tie-break: higher CV R²), then is refit on **all 60 rows**.
4. **Evaluation (Phase 5)** — **Out-of-fold** predictions for MAE, RMSE, R², residual plots, train-vs-CV gap, and feature importances when the chosen regressor exposes them.
5. **Deployment (Phase 6)** — The fitted **`growth_pipeline`** is saved to **`pipelines/growth_pipeline_v1.sav`**. FastAPI loads it when present (`GET /health` → `growth_pipeline_loaded`); score new rows with **`POST /growth/predict`** and JSON keys matching **`GROWTH_FEATURE_COLUMNS`**.

**Using new data:** Apply the same cleaning as Phase 3, build **`X_growth`** with the same six columns (still **without** `total_monetary_value`), then **`predict`**. With **N ≈ 60**, expect a **large train vs OOF error gap** for flexible models—treat outputs as **ranking and rough bands**, not exact budgets.
